# Gold-layer transaction channel summary table

This notebook creates a Gold-layer transaction channel summary table for banking analytics and reporting.

The goal is to analyze how transactions are processed across different payment gateways and device types, and to generate performance KPIs for each channel per day.

The notebook calculates:

- total transactions per day and channel
- number of successful transactions
- number of failed transactions
- average processing time per transaction

The final table created is :  banking.gold.transaction_channel_summary

In [0]:
%sql
CREATE OR REPLACE TABLE banking.gold.transaction_channel_summary AS

--  — Join Transactions with Gateway Logs
-- This joins transaction data with payment gateway logs
-- using txn_id to enrich transactions with:
-- - gateway name
-- - device type
-- - status
-- - processing time
-- Counts all transactions for each: date, gateway, device type

SELECT
    DATE(t.txn_timestamp) AS txn_date,
    pg.gateway_name,
    pg.device_type,
    COUNT(*) AS total_transactions,
    SUM(
        CASE WHEN pg.gateway_status = 'SUCCESS'
        THEN 1 ELSE 0 END
    ) AS successful_transactions,
    SUM(
        CASE WHEN pg.gateway_status = 'FAILED'
        THEN 1 ELSE 0 END
    ) AS failed_transactions,
    AVG(pg.processing_time_ms) AS avg_processing_time_ms
FROM banking.silver.transactions t

JOIN banking.silver.payment_gateway_logs pg
    ON t.txn_id = pg.txn_id
GROUP BY
    txn_date,
    pg.gateway_name,
    pg.device_type

## Validate Table Creation

In [0]:
count = spark.sql("""
SELECT COUNT(*) AS cnt
FROM banking.gold.transaction_channel_summary
""").collect()[0]["cnt"]

dbutils.notebook.exit(str(count))